<a href="https://colab.research.google.com/github/mixxr/gcolab-ml/blob/main/funnel/10_capstone_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Episode 10 — Capstone: An End-to-End Campaign
**Drug Discovery in Code** · [Life Sciences AI Applications]
This is the finish line. We chain **every** step from the series into one reproducible mini
virtual-screening campaign against **EGFR** — from raw ChEMBL data to a short, ranked, ADMET-clean
list of candidates, structurally confirmed by docking.
> Educational only — not medical advice.

## The whole pipeline, in one place
```
  ChEMBL EGFR data            (Ep 3)   ~1,065 measured inhibitors
     └─ drug-likeness filter  (Ep 4)   Ro5 + PAINS  → 843
     └─ fingerprints          (Ep 2)   Morgan 2048-bit
     └─ QSAR model            (Ep 6)   random forest, R²≈0.7
     └─ screen a library      (Ep 5/8) rank unseen molecules
     └─ ADMET gate            (Ep 9)   QED + Brenk alerts
     └─ dock survivors        (Ep 7)   smina, into real EGFR
            ▼
     ranked, clean, confirmed candidates
```

## 1. Data → features → model (Episodes 3, 4, 6)

In [ ]:
import numpy as np, pandas as pd, sys
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, QED, RDConfig
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def fp(s):
    a = np.zeros((2048,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(gen.GetFingerprint(Chem.MolFromSmiles(s)), a)
    return a

df = pd.read_csv("../data/egfr_chembl.csv")
X = np.stack(df.smiles.apply(fp)); y = df.pIC50.values
tr, te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=7)
model = RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=7).fit(X[tr], y[tr])

## 2. Screen an unseen library, then gate on ADMET (Episodes 8, 9)

In [ ]:
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
brenk = FilterCatalog(params)

lib = df.iloc[te].copy()
lib["pred_pIC50"] = model.predict(X[te])

def clean(smiles):
    m = Chem.MolFromSmiles(smiles)
    return QED.qed(m) >= 0.4 and brenk.GetMatches(m) == ()

shortlist = (lib[lib.smiles.apply(clean)]
               .sort_values("pred_pIC50", ascending=False)
               .head(10))
print(len(shortlist), "clean, high-scoring candidates")

## 3. Dock the survivors to confirm (Episode 7)
The final gate — dock each candidate into the real EGFR pocket and keep the strong binders.
```python
shortlist['dock_kcal'] = [dock(s) for s in shortlist.smiles]   # smina, from Ep 7
final = shortlist[shortlist.dock_kcal <= -8.0]
```

## The campaign result
Starting from a thousand measured molecules, the pipeline delivered a **short, ranked list of
clean candidates** that pass three independent tests — a machine-learning potency prediction, an
ADMET/toxicity screen, and physics-based docking into the real target. Every number here came out
of code you can run yourself.

![final shortlist](../images/final_table.png)

The two top PASS candidates carry **no toxicity alerts**, good drug-likeness (QED > 0.55), and dock
below **−8 kcal/mol** — exactly the profile you'd advance to synthesis and assay.

## What you built
Across ten episodes you went from `Chem.MolFromSmiles("...")` to a working, end-to-end
computational drug-discovery pipeline:
1. Load & inspect molecules · **2.** represent them (graphs, descriptors, fingerprints)
3. Pull real ChEMBL data · **4.** filter for drug-likeness · **5.** similarity search
6. Train a QSAR model · **7.** dock into a protein · **8.** screen a library at scale
9. ADMET & optimization · **10.** chain it all into one campaign

Every tool here is free and open-source: **RDKit, ChEMBL, scikit-learn, smina**. The target was
EGFR, but the *pipeline* is general — point it at another target's ChEMBL data and it runs.

**Thanks for coding along.** ⭐ the repo, clone it, and screen a target of your own.

— *Life Sciences AI Applications*